In [ ]:
import pyrootutils
import pandas as pd
import numpy as np
import networkx as nx
import community as community_louvain
import plotly.express as px
import json
from pathlib import Path
import copy
import unicodedata

# --- 1. THIẾT LẬP ROOT DỰ ÁN ---
root = pyrootutils.setup_root(
    search_from="." if Path.cwd().name != "notebooks" else "..", 
    indicator=[".git", "README.md"], 
    pythonpath=True
)

# --- 2. ĐƯỜNG DẪN FILE ---
CASE_DATA_PATH = root / "data" / "raw" / "dengue_cases_by_district_2022_2024.csv"
GEOJSON_PATH = root / "data" / "raw" / "hcmc_districts.geojson"
PROCESSED_DIR = root / "data" / "processed"
OUTPUT_DIR = root / "notebooks" / "outputs"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- 3. HÀM CHUẨN HÓA (CHÌA KHÓA FIX ANIMATION) ---
def clean_name(s):
    """Xóa dấu cách, chuẩn hóa Unicode và viết thường để khớp 100%"""
    if not isinstance(s, str): return str(s)
    s = unicodedata.normalize('NFC', s)
    return s.replace(" ", "").lower().strip()

# --- 4. NẠP VÀ XỬ LÝ DỮ LIỆU ---
df = pd.read_csv(CASE_DATA_PATH)
df['period'] = df['year'].astype(str) + '-' + df['month'].astype(str).str.zfill(2)
df['week_id'] = df['year'].astype(str) + '-W' + df['week'].astype(str).str.zfill(2)

# Xác định Spike Weeks (Tuần bùng phát)
weekly_total = df.groupby('week_id')['cases'].sum().reset_index()
threshold = weekly_total['cases'].mean() + 2 * weekly_total['cases'].std()
weekly_total['is_spike'] = weekly_total['cases'] > threshold

# --- 5. TÍNH TOÁN NETWORK METRICS ---
def compute_network_metrics(df):
    results = []
    periods = sorted(df['period'].unique())
    all_districts = sorted(df['district_name'].unique())
    
    for p in periods:
        subset = df[df['period'] == p]
        pivot = subset.pivot_table(index='week', columns='district_name', values='cases', fill_value=0)
        if pivot.shape[0] < 2: continue
            
        corr = pivot.corr().fillna(0)
        G = nx.Graph()
        for i, d1 in enumerate(pivot.columns):
            for j, d2 in enumerate(pivot.columns):
                if i < j and corr.loc[d1, d2] > 0.6:
                    G.add_edge(d1, d2, weight=corr.loc[d1, d2])
        
        cent = nx.degree_centrality(G)
        partition = community_louvain.best_partition(G) if G.edges() else {}
        
        for dist in all_districts:
            results.append({
                'period': p,
                'district': dist,
                'centrality': cent.get(dist, 0),
                'community': str(partition.get(dist, "Isolated"))
            })
    return pd.DataFrame(results)

df_metrics = compute_network_metrics(df)

# --- 6. KHỚP DỮ LIỆU VỚI GEOJSON ---
with open(GEOJSON_PATH, encoding='utf-8') as f:
    geojson_data = json.load(f)

# Tự động tìm key nhãn trong GeoJSON (name, NAME_2, v.v.)
sample_name_clean = clean_name(df_metrics['district'].iloc[0])
target_prop = "name"
for feat in geojson_data['features']:
    for k, v in feat.get('properties', {}).items():
        if clean_name(str(v)) == sample_name_clean:
            target_prop = k
            break

# Fix GeoJSON: Xóa khoảng trắng trong thuộc tính bản đồ
geojson_fixed = copy.deepcopy(geojson_data)
for f in geojson_fixed['features']:
    if target_prop in f['properties']:
        f['properties'][target_prop] = clean_name(f['properties'][target_prop])

# Fix DataFrame: Xóa khoảng trắng để khớp với GeoJSON đã fix
df_metrics['district_match'] = df_metrics['district'].apply(clean_name)

# REINDEX: Đảm bảo frame nào cũng có đủ 22 quận (Tránh lỗi mất vùng khi chạy animation)
all_periods = sorted(df_metrics['period'].unique())
all_dist_match = sorted(df_metrics['district_match'].unique())
full_idx = pd.MultiIndex.from_product([all_periods, all_dist_match], names=['period', 'district_match'])
df_final = df_metrics.set_index(['period', 'district_match']).reindex(full_idx, fill_value=0).reset_index()

# --- 7. TRỰC QUAN HÓA ---

# A. Biểu đồ Spike Weeks
fig_spike = px.bar(weekly_total, x='week_id', y='cases', color='is_spike',
                   title="Phân tích các tuần bùng phát dịch (Spike Weeks)",
                   color_discrete_map={True: '#ef5350', False: '#26a69a'},
                   template='plotly_white')
fig_spike.add_hline(y=threshold, line_dash="dash", line_color="red")

# B. Community Stability (Số lượng cụm lây nhiễm)
comm_data = df_final[df_final['community'] != '0'].groupby('period')['community'].nunique().reset_index()
fig_comm = px.line(comm_data, x='period', y='community', markers=True,
                   title="Sự thay đổi số lượng cộng đồng dịch tễ (Clusters)",
                   template='plotly_white')

# C. Animation Map
fig_map = px.choropleth(
    df_final,
    geojson=geojson_fixed,
    locations='district_match',
    featureidkey=f"properties.{target_prop}",
    color='centrality',
    animation_frame='period',
    color_continuous_scale="YlOrRd",
    range_color=[0, 0.7],
    title="Diễn biến Centrality Sốt xuất huyết TP.HCM (2022-2024)",
    labels={'centrality': 'Mức độ trọng điểm', 'district_match': 'Mã Quận'}
)
fig_map.update_geos(fitbounds="locations", visible=False)
fig_map.update_layout(margin={"r":0,"t":50,"l":0,"b":0})

# --- 8. LƯU VÀ HIỂN THỊ ---
fig_spike.show()
fig_comm.show()
fig_map.show()

fig_map.write_html(OUTPUT_DIR / "final_graph_analysis.html")